# Day 10 阶段复习与购物车项目

**前置**: Day01-09 全部语法点(print/input/变量/字符串/分支/循环/函数/列表字典/文件 I/O/异常)
**关键问题**: 前 9 天学了语法点,今天把它们焊接成一个完整控制台应用,同时歼灭高频错误
**今日节奏**: 4 类高频错题 -> 读错方法论 -> 购物车 v1(函数式) -> JSON 持久化 -> 购物车 v2(OOP)

# 高频错误 1:if/elif 边界错误

错误写法:`elif score >= 80` 写在 `elif score >= 60` 之后永远执行不到,因为 >= 60 先吃掉所有 >= 60 的分数。正确做法是从高到低判断:先 >= 80, 再 >= 60。

In [ ]:
# ===== 典型的 if/elif 边界错误 =====
score = 85

# 错误写法:低分条件在前,高分永远进不来
if score >= 60:
    print("及格")            # 85 分也只会输出 及格
elif score >= 80:
    print("良好")            # 永远执行不到

# 正确写法:高分条件在前,从高到低判断
if score >= 80:
    print("良好")            # 正确执行到这里
elif score >= 60:
    print("及格")

# 高频错误 2:累加器在循环内重新初始化

错误写法:`total = 0` 写在循环里,每次迭代都被清零,结果只加到最后一个值。正确做法是累加器在循环外初始化,循环内只累加。

In [ ]:
# ===== 累加器位置错误 =====
prices = [5.5, 12.8, 8.0]

# 错误写法:total 在循环内被反复清零
total_wrong = 0
for p in prices:
    total_wrong = 0       # 每次循环都被清零
    total_wrong += p
print(f"错误结果: {total_wrong}")   # 只保留了最后一个值

# 正确写法:total 在循环外初始化
total_right = 0
for p in prices:
    total_right += p
print(f"正确结果: {total_right}")

# 高频错误 3:return 与 print 混淆

错误写法:函数里 print 了却没有 return,调用者拿到的永远是 None。正确做法是用 return 把值交出去,打印由调用者决定。同理注意 input() 返回字符串,和数字比较需 int() 转换,否则抛 TypeError。

In [ ]:
# ===== return vs print + input() 返回类型 =====
def calc_print(nums):
    print(sum(nums))      # 屏幕上能看到,但调用者拿不到

def calc_return(nums):
    return sum(nums)      # 调用者拿到返回值

data = [5.5, 12.8, 8.0]
r1 = calc_print(data)
print(f"calc_print 返回值: {r1}")    # None
r2 = calc_return(data)
print(f"calc_return 返回值: {r2}")   # 26.3

# input() 返回字符串演示
user_input = "25"
print(f"input 返回类型: {type(user_input)}")   # <class 'str'>
# 实际项目用 try/except 包裹 int() 转换
def safe_int(prompt):
    try:
        return int(input(prompt))
    except ValueError:
        print("请输入数字!")
        return None

# 读错方法论

遇到报错不要慌张,按以下顺序读错误信息:

1. 看最后一行异常类型(ValueError/TypeError/KeyError)
2. 看报错行号,定位出错代码
3. 用 `print(type(变量))` + `print(变量)` 打印变量类型与值
4. 对照边界条件(空列表/零/负数/单元素)

这套方法论是后续排查购物车 Bug 的核心工具。

# 购物车需求分析与数据结构设计

需求 checklist:商品列表(字典 + 列表) / 菜单循环 / 同商品数量累加 / 结算 + 总价 / 异常加固 / JSON 持久化。

数据字典约定:商品 = {"id": int, "name": str, "price": float},购物车项 = {"id": int, "name": str, "price": float, "qty": int}

In [ ]:
# ===== 商品库与购物车数据结构 =====
products = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]

# 购物车每项在商品基础上增加 qty 字段
cart = [
    {"id": 1, "name": "苹果", "price": 5.50, "qty": 2},
    {"id": 3, "name": "面包", "price": 8.00, "qty": 1},
]

for item in cart:
    小计 = item["price"] * item["qty"]
    print(f"{item['name']} x{item['qty']} = {小计:.2f}")

# 菜单循环 + main() 调度框架

口诀:框架先把函数名填上,每个函数只做一件事,main 只做调度。while True 负责循环,break 负责退出。

In [ ]:
# ===== 主循环调度框架 =====
def browse_products():
    pass

def add_to_cart():
    pass

def view_cart():
    pass

def checkout():
    pass

def main():
    while True:
        print("\n1.浏览商品 2.加入购物车 3.查看购物车 4.结算 0.退出")
        choice = input("请选择:")
        if choice == "0":
            print("再见!")
            break
        elif choice == "1":
            browse_products()
        elif choice == "2":
            add_to_cart()
        elif choice == "3":
            view_cart()
        elif choice == "4":
            checkout()
        else:
            print("无效选项!")

if __name__ == "__main__":
    main()

# find_product() 与 browse_products() 辅助函数

find_product():按 id 在商品库查找商品,找到返回字典找不到返回 None。browse_products():遍历商品库打印每条信息。两个函数配合后续的 add_to_cart 使用。

In [ ]:
# ===== find_product + browse_products =====
products = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]

def find_product(pid):
    # 遍历商品库,找到 id 匹配的商品返回字典
    for p in products:
        if p["id"] == pid:
            return p
    return None            # 找不到返回 None

def browse_products():
    print("--- 商品列表 ---")
    for p in products:
        print(f"{p['id']:>2}. {p['name']:<6} ￥{p['price']:.2f}")
    print("----------------")

# 测试两个函数
browse_products()
result = find_product(2)
if result:
    print(f"找到: {result['name']}")
print(f"不存在的查询: {find_product(99)}")    # None

# add_to_cart() 加入购物车 + 异常处理

核心逻辑:输入 id 转整数,找到商品后判断购物车是否已有,有则 qty + 1,无则新增条目。try/except 包裹 int() 转换防止非法输入崩溃。

In [ ]:
# ===== add_to_cart() 加入购物车 =====
products = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]
cart = []

def find_product(pid):
    for p in products:
        if p["id"] == pid:
            return p
    return None

def add_to_cart():
    try:
        pid = int(input("请输入商品编号:"))
    except ValueError:
        print("请输入数字!")
        return
    p = find_product(pid)
    if p is None:
        print("商品不存在!")
        return
    for item in cart:
        if item["id"] == pid:
            item["qty"] += 1
            print(f"{p['name']} 数量 +1")
            return
    cart.append({"id": pid, "name": p["name"],
                 "price": p["price"], "qty": 1})
    print(f"已加入: {p['name']}")

# 用 StringIO 模拟输入演示
import sys
from io import StringIO
sys.stdin = StringIO("2\n2\n1\n")
add_to_cart()
add_to_cart()
add_to_cart()
sys.stdin = sys.__stdin__
print(f"当前购物车: {cart}")

# view_cart() 与 checkout() 查看与结算

view_cart():遍历购物车打印每条小计与总价,注意累加器 total 在循环外初始化。checkout():查看 + 清空购物车,用 `cart.clear()` 原地清空不要用 `cart = []`。

In [ ]:
# ===== view_cart + checkout =====
cart = [
    {"id": 1, "name": "苹果", "price": 5.50, "qty": 2},
    {"id": 3, "name": "面包", "price": 8.00, "qty": 1},
]

def view_cart():
    if not cart:
        print("购物车为空!")
        return
    print("--- 购物车 ---")
    total = 0                       # 累加器在循环外初始化
    for item in cart:
        小计 = item["price"] * item["qty"]
        print(f"{item['name']} x{item['qty']} = ￥{小计:.2f}")
        total += 小计
    print(f"总价: ￥{total:.2f}")

def checkout():
    if not cart:
        print("购物车为空!")
        return
    view_cart()
    cart.clear()                    # 原地清空,不要用 cart = []
    print("已清空购物车")

checkout()
print(f"结算后: {cart}")

# JSON 持久化:save_cart() 与 load_cart()

程序退出后数据不能丢,用 json.dump() 写入文件,json.load() 读取文件。`ensure_ascii=False` 保证中文,`indent=2` 让文件可读。load 时用 try/except 捕获 JSONDecodeError 防崩溃。

In [ ]:
# ===== save_cart + load_cart =====
import json, os

cart = [{"id": 1, "name": "苹果", "price": 5.5, "qty": 2}]
FILE = "cart.json"

def save_cart():
    with open(FILE, "w", encoding="utf-8") as f:
        # ensure_ascii=False 保证中文正常显示
        json.dump(cart, f, ensure_ascii=False, indent=2)
    print("购物车已保存")

def load_cart():
    if not os.path.exists(FILE):
        return []
    with open(FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            print("文件损坏,重置购物车")
            return []

save_cart()
loaded = load_cart()
print(f"加载结果: {loaded}")

# 完整整合:启动加载 + 运行时保存

把 load 和 save 串入购物车生命周期:启动时 load,每次操作后 save。即使程序异常退出数据也不丢失。

In [ ]:
# ===== 完整整合 Demo =====
import json, os

products = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]
FILE = "cart.json"

def save_cart():
    with open(FILE, "w", encoding="utf-8") as f:
        json.dump(cart, f, ensure_ascii=False, indent=2)

def load_cart():
    if not os.path.exists(FILE):
        return []
    with open(FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

cart = load_cart()                        # 启动加载
print(f"启动加载: {cart}")
cart.append({"id": 1, "name": "苹果",
             "price": 5.5, "qty": 1})
save_cart()                               # 操作后保存
cart2 = load_cart()                       # 模拟重启
print(f"重启加载: {cart2}")

# 购物车 v2:OOP 版 Product 与 Cart 类

把商品抽象成类,每个商品是一个对象。类构造函数 __init__ 初始化属性,__str__ 定义打印格式。把购物车也抽象成类,封装 items 列表和 add / total / checkout 方法,数据和操作绑定职责更清晰。

In [ ]:
# ===== Product 类 + Cart 类 =====
class Product:
    def __init__(self, pid, name, price):
        self.pid = pid
        self.name = name
        self.price = price

    def __str__(self):
        # 定义对象的打印格式
        return f"{self.pid:>2}. {self.name:<6} ￥{self.price:.2f}"

class Cart:
    def __init__(self):
        # 每个购物车对象有自己的 items 列表
        self.items = []           # 每项: {"product": Product, "qty": int}

    def add(self, product, qty=1):
        # 判断购物车是否已有该商品
        for item in self.items:
            if item["product"].pid == product.pid:
                item["qty"] += qty
                return
        # 没有则新增条目
        self.items.append({"product": product, "qty": qty})

    def total(self):
        # 计算总价,累加器在循环外初始化
        t = 0
        for item in self.items:
            t += item["product"].price * item["qty"]
        return t

    def view(self):
        # 打印购物车明细
        for item in self.items:
            p = item["product"]
            小计 = p.price * item["qty"]
            print(f"{p.name} x{item['qty']} = ￥{小计:.2f}")
        print(f"总价: ￥{self.total():.2f}")

    def checkout(self):
        # 结算并清空购物车
        self.view()
        self.items.clear()

p1 = Product(1, "苹果", 5.50)
p2 = Product(2, "牛奶", 12.80)
print(p1)                       # 自动调用 __str__
cart = Cart()
cart.add(p1, 2)
cart.add(p2, 1)
cart.view()

# OOP 整合 Demo

把 Product 和 Cart 串入 main 循环,实现面向对象版购物车。对比函数式版本,OOP 把数据和行为绑定,扩展(如打折、会员)更容易。

In [ ]:
# ===== OOP 整合运行 =====
class Product:
    def __init__(self, pid, name, price):
        self.pid = pid
        self.name = name
        self.price = price

class Cart:
    def __init__(self):
        self.items = []

    def add(self, product, qty=1):
        for item in self.items:
            if item["product"].pid == product.pid:
                item["qty"] += qty
                return
        self.items.append({"product": product, "qty": qty})

    def total(self):
        t = 0
        for item in self.items:
            t += item["product"].price * item["qty"]
        return t

    def view(self):
        if not self.items:
            print("购物车为空!")
            return
        for item in self.items:
            p = item["product"]
            小计 = p.price * item["qty"]
            print(f"{p.name} x{item['qty']} = ￥{小计:.2f}")
        print(f"总价: ￥{self.total():.2f}")

    def checkout(self):
        if not self.items:
            print("购物车为空!")
            return
        self.view()
        self.items.clear()

products = [
    Product(1, "苹果", 5.50),
    Product(2, "牛奶", 12.80),
    Product(3, "面包", 8.00),
]
cart = Cart()
cart.add(products[0], 2)
cart.add(products[1], 1)
cart.add(products[0], 1)            # 苹果累计 x3
cart.view()
print("--- 结算 ---")
cart.checkout()
print(f"结算后条目数: {len(cart.items)}")

# 今日小结

把 Day01-09 语法焊接成完整控制台项目。

四大高频错:if/elif 从高到低写 / 累加器在循环外 / return 把值交出去 / input() 返回字符串要 int() 转。

技巧:函数内改列表用 .clear() / .append() 等原地方法,不要赋值 lst = [];全局变量在函数内只读不需要 global,要改才需要声明 global。

购物车版本演进:v1 函数式 -> 加 JSON 持久化 -> v2 OOP。OOP 把数据和行为绑定,扩展更容易。

# 更多练习

当堂练:lesson10/in_class/practice01.py ~ practice06.py
课后作业:lesson10/homework/task01.py ~ task03.py
参考资料:summary.md Day10 进度说明